# TEKNOFEST Healthcare AI — Phase 2.5: Preprocessing Audit
## Is Preprocessing Necessary? What Exactly Needs to Change?

**Answer**: YES, preprocessing is necessary, but MINIMAL for tree-based baseline.
The key improvements are: constant column removal, missingness indicators, categorical encoding,
and class weight balancing.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings
from pathlib import Path; from collections import OrderedDict
warnings.filterwarnings('ignore'); np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (14,6); plt.rcParams['font.size'] = 11

DATA = Path("EĞİTİM (TRAIN) SETLERİ 2")
ds = OrderedDict()
for f in sorted(DATA.glob("*.csv")):
    k = f.stem.replace("YARISMA_TRAIN_","")
    ds[k] = pd.read_csv(f)
    print(f"{k}: {ds[k].shape}")
M = ds["MASTER"]
al = [c for c in M.columns if c.startswith("AL_")]
ek = [c for c in M.columns if c.startswith("EK_")]
cat = [c for c in M.columns if c.startswith("CAT_")]
aa = [c for c in M.columns if c.startswith("AA_")]
print(f"\nAL: {len(al)}, CAT: {len(cat)}, EK: {len(ek)}, AA: {len(aa)}")


## B. Data Readiness Verdicts

In [ ]:
verdict_df = pd.read_csv("reports/phase_02_5_preprocessing_audit/data_readiness_verdict.csv")
display(verdict_df)


## C. Missing Value Analysis

In [ ]:
# Row-level missingness by class
fig, axes = plt.subplots(1,4, figsize=(18,4))
for idx,(k,df) in enumerate(ds.items()):
    rm = df[al+ek].isnull().mean(axis=1)*100
    for lab,col,nm in [(0,'#2196F3','Benign'),(1,'#F44336','Pathogenic')]:
        sub = rm[df.Label==lab]
        axes[idx].hist(sub, bins=30, alpha=.6, color=col, label=nm, density=True)
    axes[idx].set_title(f"{k} (n={len(df)})"); axes[idx].set_xlabel("Row Miss %"); axes[idx].legend(fontsize=8)
plt.suptitle("Row-Level Missingness by Class", fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Label-associated missingness (MASTER top 20)
lam = pd.read_csv("reports/phase_02_5_preprocessing_audit/label_associated_missingness.csv")
top20 = lam.head(20)
fig, ax = plt.subplots(figsize=(12,6))
ax.barh(range(len(top20)), top20.AbsDiff.values, color='#FF9800')
ax.set_yticks(range(len(top20))); ax.set_yticklabels(top20.Column.values, fontsize=8)
ax.set_xlabel("Abs Diff in Miss% (Pathogenic vs Benign)"); ax.invert_yaxis()
ax.set_title("Top 20: Label-Associated Missingness (MASTER)"); plt.tight_layout(); plt.show()


## D. Duplicate & Label Noise

In [ ]:
dup_df = pd.read_csv("reports/phase_02_5_preprocessing_audit/duplicate_audit.csv")
display(dup_df)
print("\n1 conflicting label in MASTER, 1 in PAH — accept as label noise.")


## E. Constant & Low-Information Features

In [ ]:
# Count constant columns per dataset
for k,df in ds.items():
    nc = sum(df[c].nunique(dropna=True)<=1 for c in df.columns if c not in ['Variant_ID','Label'])
    print(f"{k}: {nc} constant columns")

# Global constants
const_df = pd.read_csv("reports/phase_02_5_preprocessing_audit/constant_low_information_features.csv")
print(f"\nGlobal constants (to DROP): {(const_df.Handling=='DROP').sum()}")
print(f"Binary features: {(const_df.SuspectedType=='binary').sum()}")
print(f"Quasi-constant: {(const_df.SuspectedType=='quasi-constant').sum()}")


## F. EK Feature Risk Analysis

In [ ]:
# EK inter-correlation
ek_corr = M[ek].corr(method='spearman')
fig, ax = plt.subplots(figsize=(8,7))
sns.heatmap(ek_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("EK Feature Inter-Correlation (Spearman)"); plt.tight_layout(); plt.show()

# EK by class
fig, axes = plt.subplots(3,3, figsize=(16,12))
for idx,c in enumerate(ek):
    ax = axes[idx//3][idx%3]
    for lab,col,nm in [(0,'#2196F3','Benign'),(1,'#F44336','Pathogenic')]:
        sub = M.loc[M.Label==lab,c].dropna()
        if len(sub)>10: sub.plot(kind='kde', ax=ax, color=col, label=nm, alpha=.7)
    corr = M[['Label',c]].dropna().corr().iloc[0,1]
    ax.set_title(f"{c} (corr={corr:.3f})"); ax.legend(fontsize=8)
plt.suptitle("EK Features by Class", fontweight='bold'); plt.tight_layout(); plt.show()


## G. Encoding Recommendations

In [ ]:
enc_df = pd.read_csv("reports/phase_02_5_preprocessing_audit/encoding_recommendations.csv")
display(enc_df)


## H. Preprocessing Pipelines

| Pipeline | Purpose | Key Steps |
|----------|---------|-----------|
| **A: Minimal GBDT** | Fastest baseline | Drop constants + Variant_ID; class weights; native missing |
| **B: Missingness-Aware GBDT** | Main workhorse | A + miss indicators + CAT/AA encoding |
| **C: Anti-Leakage GBDT** | Circularity check | B minus EK_4/5/6 |
| **D: Linear Baseline** | Calibration reference | Impute + scale + one-hot + LogisticRegression |
| **E: Ensemble Prep** | Final performance | B + OOF stacking + panel calibration |

**Phase 3 should start with Pipeline A (Day 1), then Pipeline B (Day 2), then Pipeline C (Day 3).**


## Executive Summary

### Is Preprocessing Needed? **YES**

### Mandatory:
1. Drop Variant_ID, drop 57 constant columns
2. Class weights (73% pathogenic)
3. Stratified CV

### Recommended:
4. Missingness indicators (binary per feature + row count)
5. CAT_3/4/5 one-hot, CAT_1/2 label encode, CAT_6 binary flag
6. AA_1/AA_2 one-hot
7. EK ablation (with/without EK_4/5/6)

### AVOID:
- Imputation for GBDT models
- SMOTE/oversampling
- Feature scaling for GBDT
- Target encoding outside CV folds
